# Assignment 1: Network Structure Analysis
**CIEQ6232 — Academic year 2025/26 — Q4**

City: **Prague** | Mode: **Subway**

In [ ]:
from bokeh.resources import INLINE
import bokeh.io
bokeh.io.output_notebook(INLINE)

from gtfspy import gtfs
import networkx as nx
import pickle
import utils

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path

plt.rcParams['figure.dpi'] = 110
Path('output').mkdir(exist_ok=True)

---
## Part 1 — Build & clean the graph

In [ ]:
# ── Config ──────────────────────────────────────────────────
NETWORK = 'prague'
MODE    = 'Subway'
DB_PATH = f'./{NETWORK}.sqlite'
# ────────────────────────────────────────────────────────────

g = gtfs.GTFS(DB_PATH)
print('Available modes:', [utils.mode_name[x] for x in g.get_modes()])

In [ ]:
# Generate raw L-space
L = utils.generate_graph(g, MODE, start_hour=5, end_hour=24)
print(f'Raw L-space  — nodes: {L.number_of_nodes()}  edges: {L.number_of_edges()}')

In [ ]:
utils.plot_graph(L, back_map='OSM')

### 1.1 — Clean: merge stops with same name (delta = 200 m)

In [ ]:
L_merged = utils.merge_stops_with_same_name(L, delta=200)
print(f'After name-merge — nodes: {L_merged.number_of_nodes()}  edges: {L_merged.number_of_edges()}')

### 1.2 — Clean: remove disconnected islands

In [ ]:
utils.check_islands(L_merged)

### 1.3 — Clean: merge recommender  
*Run round 1 (exact overlaps ≤ 20 m), then round 2 (similar names ≤ 500 m).  
Each candidate is shown on a map — type **y** to merge, **n** to skip.*

In [ ]:
utils.merge_recommender(L_merged, string_match=0, stop_distance=20)

In [ ]:
utils.merge_recommender(L_merged, string_match=75, stop_distance=500)

### 1.4 — Sanity check

In [ ]:
utils.sanity_check(L_merged)
utils.plot_graph(L_merged, back_map='OSM')

### 1.5 — Save cleaned L-space

In [ ]:
L_space_path = f'./{NETWORK}.pkl'
G_int = nx.convert_node_labels_to_integers(L_merged)
with open(L_space_path, 'wb') as f:
    pickle.dump(G_int, f)
print(f'Saved {L_space_path}')

---
## Part 2 — Build P-space

In [ ]:
# Reload
g = gtfs.GTFS(DB_PATH)
with open(L_space_path, 'rb') as f:
    L_graph = pickle.load(f)

P = utils.P_space(g, L_graph, mode=MODE)
print(f'P-space — nodes: {P.number_of_nodes()}  edges: {P.number_of_edges()}')

In [ ]:
utils.plot_graph(P, space='P', back_map='OSM')

In [ ]:
# Save both graphs together
network_bundle = {'L_space': L_graph, 'P_space': P, 'city': NETWORK, 'mode': MODE}
with open('output/Network.pkl', 'wb') as f:
    pickle.dump(network_bundle, f)
print('Saved output/Network.pkl')

---
## Part 3 — Global network indicators

In [ ]:
# Convert DiGraph → undirected for indicator calculations
L_ud = L_graph.to_undirected()
P_ud = P.to_undirected()

# Keep largest connected component (should already be 1 CC after cleaning)
L_ud = L_ud.subgraph(max(nx.connected_components(L_ud), key=len)).copy()
P_ud = P_ud.subgraph(max(nx.connected_components(P_ud), key=len)).copy()

N_L, E_L = L_ud.number_of_nodes(), L_ud.number_of_edges()
N_P, E_P = P_ud.number_of_nodes(), P_ud.number_of_edges()

print('=' * 50)
print('GLOBAL INDICATORS')
print('=' * 50)
print(f'Nodes  — L-space: {N_L}   P-space: {N_P}')
print(f'Edges  — L-space: {E_L}   P-space: {E_P}')

In [ ]:
print('Computing diameter and ASP (may take a moment)...')

diam_L_uw = nx.diameter(L_ud)
diam_L_w  = nx.diameter(L_ud, weight='duration_avg')
diam_P_uw = nx.diameter(P_ud)
diam_P_w  = nx.diameter(P_ud, weight='avg_wait')

asp_L_uw = nx.average_shortest_path_length(L_ud)
asp_L_w  = nx.average_shortest_path_length(L_ud, weight='duration_avg')
asp_P_uw = nx.average_shortest_path_length(P_ud)
asp_P_w  = nx.average_shortest_path_length(P_ud, weight='avg_wait')

# Connectivity (gamma index) — L-space, planar formula
gamma = E_L / (3 * (N_L - 2))
# Meshedness (alpha index) — L-space
alpha = (E_L - N_L + 1) / (2 * N_L - 5)

print(f'\nDiameter  L unweighted : {diam_L_uw} hops')
print(f'Diameter  L weighted   : {diam_L_w:.1f} s')
print(f'Diameter  P unweighted : {diam_P_uw} hops')
print(f'Diameter  P weighted   : {diam_P_w:.2f} min')
print(f'\nASP       L unweighted : {asp_L_uw:.4f} hops')
print(f'ASP       L weighted   : {asp_L_w:.1f} s')
print(f'ASP       P unweighted : {asp_P_uw:.4f} hops')
print(f'ASP       P weighted   : {asp_P_w:.4f} min')
print(f'\nGamma (connectivity)   : {gamma:.4f}')
print(f'Alpha (meshedness)     : {alpha:.4f}')

global_df = pd.DataFrame({
    'Indicator': [
        'N (L)', 'N (P)', 'E (L)', 'E (P)',
        'Diameter L unweighted (hops)', 'Diameter L weighted (s)',
        'Diameter P unweighted (hops)', 'Diameter P weighted (min)',
        'ASP L unweighted (hops)', 'ASP L weighted (s)',
        'ASP P unweighted (hops)', 'ASP P weighted (min)',
        'Gamma', 'Alpha',
    ],
    'Value': [
        N_L, N_P, E_L, E_P,
        diam_L_uw, diam_L_w, diam_P_uw, diam_P_w,
        asp_L_uw, asp_L_w, asp_P_uw, asp_P_w,
        gamma, alpha,
    ]
})
global_df.to_csv('output/global_indicators.csv', index=False)
print('\nSaved output/global_indicators.csv')
global_df

---
## Part 4 — Local indicators

In [ ]:
def local_stats(c_dict, G, label, top=3):
    vals = np.array(list(c_dict.values()))
    top_nodes = sorted(c_dict, key=c_dict.get, reverse=True)[:top]
    top_list  = [(G.nodes[n].get('name', str(n)), round(c_dict[n], 6)) for n in top_nodes]
    print(f'\n{label}')
    print(f'  mean={vals.mean():.6f}  std={vals.std():.6f}  '
          f'min={vals.min():.6f}  max={vals.max():.6f}')
    print(f'  top {top}: {top_list}')
    return {'mean': vals.mean(), 'std': vals.std(),
            'min': vals.min(), 'max': vals.max(), f'top_{top}': top_list}

### 4.1 — L-space local indicators

In [ ]:
deg_L       = nx.degree_centrality(L_ud)
close_L_uw  = nx.closeness_centrality(L_ud)
close_L_w   = nx.closeness_centrality(L_ud, distance='duration_avg')

print('Computing betweenness (L-space)...')
between_L_uw = nx.betweenness_centrality(L_ud, normalized=True)
between_L_w  = nx.betweenness_centrality(L_ud, weight='duration_avg', normalized=True)

local_stats(deg_L,        L_ud, 'Degree centrality — L-space')
local_stats(close_L_uw,   L_ud, 'Closeness centrality — L-space unweighted')
local_stats(close_L_w,    L_ud, 'Closeness centrality — L-space weighted')
local_stats(between_L_uw, L_ud, 'Betweenness centrality — L-space unweighted')
local_stats(between_L_w,  L_ud, 'Betweenness centrality — L-space weighted')

### 4.2 — P-space local indicators

In [ ]:
deg_P       = nx.degree_centrality(P_ud)
close_P_uw  = nx.closeness_centrality(P_ud)
close_P_w   = nx.closeness_centrality(P_ud, distance='avg_wait')

print('Computing betweenness (P-space)...')
between_P_uw = nx.betweenness_centrality(P_ud, normalized=True)
between_P_w  = nx.betweenness_centrality(P_ud, weight='avg_wait', normalized=True)

local_stats(deg_P,        P_ud, 'Degree centrality — P-space')
local_stats(close_P_uw,   P_ud, 'Closeness centrality — P-space unweighted')
local_stats(close_P_w,    P_ud, 'Closeness centrality — P-space weighted')
local_stats(between_P_uw, P_ud, 'Betweenness centrality — P-space unweighted')
local_stats(between_P_w,  P_ud, 'Betweenness centrality — P-space weighted')

### 4.3 — Save all centrality values

In [ ]:
node_list = list(L_ud.nodes())
df = pd.DataFrame(index=node_list)
df['stop_name']     = [L_ud.nodes[n].get('name', str(n)) for n in node_list]
df['deg_L']         = pd.Series(deg_L)
df['close_L_uw']    = pd.Series(close_L_uw)
df['close_L_w']     = pd.Series(close_L_w)
df['between_L_uw']  = pd.Series(between_L_uw)
df['between_L_w']   = pd.Series(between_L_w)
df['deg_P']         = pd.Series(deg_P)
df['close_P_uw']    = pd.Series(close_P_uw)
df['close_P_w']     = pd.Series(close_P_w)
df['between_P_uw']  = pd.Series(between_P_uw)
df['between_P_w']   = pd.Series(between_P_w)

df.to_csv('output/centrality_indicators.csv')
print('Saved output/centrality_indicators.csv')
df.describe()

---
## Part 5 — Histograms

In [ ]:
centrality_sets = [
    (deg_L,        'Degree — L-space'),
    (close_L_uw,   'Closeness L (unweighted)'),
    (close_L_w,    'Closeness L (weighted)'),
    (between_L_uw, 'Betweenness L (unweighted)'),
    (between_L_w,  'Betweenness L (weighted)'),
    (deg_P,        'Degree — P-space'),
    (close_P_uw,   'Closeness P (unweighted)'),
    (close_P_w,    'Closeness P (weighted)'),
    (between_P_uw, 'Betweenness P (unweighted)'),
    (between_P_w,  'Betweenness P (weighted)'),
]

fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for ax, (c_dict, title) in zip(axes.flat, centrality_sets):
    vals = list(c_dict.values())
    ax.hist(vals, bins=20, color='steelblue', edgecolor='white', linewidth=0.4)
    ax.set_title(title, fontsize=8)
    ax.set_xlabel('Value', fontsize=7)
    ax.set_ylabel('Count', fontsize=7)
    ax.tick_params(labelsize=6)

plt.suptitle('Prague Subway — Centrality Histograms', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('output/histograms.png', bbox_inches='tight')
plt.show()
print('Saved output/histograms.png')

---
## Part 6 — Map visualisations (weighted L-space, via Bokeh)

In [ ]:
# Set centrality values as node attributes so plot_graph can colour by them
nx.set_node_attributes(L_graph, deg_L,        'degree_centrality')
nx.set_node_attributes(L_graph, close_L_w,    'closeness_w')
nx.set_node_attributes(L_graph, between_L_w,  'betweenness_w')

In [ ]:
# Degree centrality map
utils.plot_graph(L_graph, back_map='OSM', color_by='degree_centrality',
                 export_name='output/map_degree_L')

In [ ]:
# Weighted closeness centrality map
utils.plot_graph(L_graph, back_map='OSM', color_by='closeness_w',
                 export_name='output/map_closeness_L_w')

In [ ]:
# Weighted betweenness centrality map (edge colour by travel time)
utils.plot_graph(L_graph, back_map='OSM', color_by='betweenness_w',
                 edge_color_by='duration_avg',
                 export_name='output/map_betweenness_L_w')

---
## Part 7 — Pearson correlations

**Student #1** — correlations *within* a representation (all L-space indicator pairs).  
**Student #2** — correlations *across* representations (L-space vs P-space pairs).

In [ ]:
lspace_cols = ['deg_L', 'close_L_uw', 'close_L_w', 'between_L_uw', 'between_L_w']
cross_cols  = ['deg_L', 'close_L_w', 'between_L_w', 'deg_P', 'close_P_w', 'between_P_w']

corr_L     = df[lspace_cols].corr(method='pearson')
corr_cross = df[cross_cols].corr(method='pearson')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(corr_L, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, ax=axes[0])
axes[0].set_title('Pearson correlations — within L-space', fontsize=11)

sns.heatmap(corr_cross, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, ax=axes[1])
axes[1].set_title('Pearson correlations — L-space vs P-space', fontsize=11)

plt.tight_layout()
plt.savefig('output/pearson_correlations.png', bbox_inches='tight')
plt.show()
print('Saved output/pearson_correlations.png')

---
## Part 8 — Vulnerability & Redundancy indicators

Define your own composite indicators below. The placeholder formulas are suggestions — replace with your reasoned choice and justify in the report.

In [ ]:
# ── Vulnerability ────────────────────────────────────────────
# Placeholder: concentration of weighted betweenness
# (max / mean ratio — higher = more dominated by single nodes = more vulnerable)
bc_vals = np.array(list(between_L_w.values()))
vulnerability = bc_vals.max() / bc_vals.mean()

# ── Redundancy ───────────────────────────────────────────────
# Placeholder: alpha index (circuit richness — alternative paths)
redundancy = alpha

print(f'Vulnerability : {vulnerability:.4f}')
print(f'Redundancy    : {redundancy:.4f}')
print('\n[Replace these formulas with your own — justify in the report]')

In [ ]:
# ── Scatter plot (add peer networks once Brightspace data is shared) ──
# Load shared CSV from Brightspace: columns [city, vulnerability, redundancy]
# df_all = pd.read_csv('data/all_networks_indicators.csv')

df_all = pd.DataFrame([{'city': 'Prague', 'vulnerability': vulnerability, 'redundancy': redundancy}])

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(df_all['redundancy'], df_all['vulnerability'], s=80, color='steelblue', zorder=3)
for _, row in df_all.iterrows():
    ax.annotate(row['city'], (row['redundancy'], row['vulnerability']),
                textcoords='offset points', xytext=(6, 3), fontsize=9)

ax.set_xlabel('Redundancy', fontsize=11)
ax.set_ylabel('Vulnerability', fontsize=11)
ax.set_title('Vulnerability vs Redundancy — all networks', fontsize=12)
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('output/vulnerability_redundancy_scatter.png', bbox_inches='tight')
plt.show()
print('Saved output/vulnerability_redundancy_scatter.png')